# SNN baseline on CIFAR-10 / CIFAR-100 — GPU run

Runs the surrogate-gradient SNN baseline (`neuron_baseline`) on CIFAR-10 and
CIFAR-100 on an NVIDIA GPU, end to end. Works on a local CUDA machine or on
Google Colab (**Runtime → Change runtime type → GPU**). Then **Run all**.

Each run writes `runs/<dataset>-<timestamp>/` with `config.json`, `results.json`
(test accuracy, history, wall time, GPU energy), checkpoints and plots.

## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
assert torch.cuda.is_available(), "No CUDA GPU found. On Colab: Runtime -> Change runtime type -> GPU."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Get the code

- **Already inside the repo** (local machine, notebook opened from
  `snn-surrogate-baseline/notebooks/`): this cell just moves to `snn-surrogate-baseline/`.
- **Colab:** it clones the `shubhanandan` branch. If the repo is private, upload
  the `NeuroComputing` folder (or a zip of it) to `/content` first instead.

In [ ]:
import os, pathlib
here = pathlib.Path.cwd()
if (here / "neuron_baseline").is_dir():
    root = here
elif (here.parent / "neuron_baseline").is_dir():
    root = here.parent
else:
    if not pathlib.Path("/content/NeuroComputing").exists():
        subprocess.run(["git", "clone", "-b", "shubhanandan",
                        "https://github.com/alannadels/NeuroComputing.git",
                        "/content/NeuroComputing"], check=True)
    root = pathlib.Path("/content/NeuroComputing/snn-surrogate-baseline")
os.chdir(root)
print("working in:", pathlib.Path.cwd())

## 3. Install the extra packages (PyTorch is already on Colab / your CUDA env)

In [ ]:
%pip install -q "snntorch==1.0.0" nvidia-ml-py matplotlib

## 4. Smoke test (≈10 s) — checks MNIST and CIFAR-sized networks on the GPU

In [ ]:
!python -m neuron_baseline smoke

## 5. Train

`EPOCHS` and `LR` are the main knobs. Tutorial 6 used `lr = 1e-2` for MNIST;
for CIFAR also try `1e-3`. CIFAR downloads automatically into `data/` on the
first run (~170 MB each).

In [ ]:
EPOCHS = 20
LR = 1e-2
WORKERS = 2          # Colab has 2 CPU cores; use 4-8 on a desktop

In [ ]:
!python -m neuron_baseline train --dataset CIFAR10 --epochs {EPOCHS} --lr {LR} --num-workers {WORKERS} --device cuda

In [ ]:
!python -m neuron_baseline train --dataset CIFAR100 --epochs {EPOCHS} --lr {LR} --num-workers {WORKERS} --device cuda

## 6. Results

In [ ]:
import json, glob
from IPython.display import Image, display
rows = []
for d in sorted(glob.glob("runs/cifar*-*")):
    r = pathlib.Path(d, "results.json")
    if not r.exists():
        continue
    res = json.loads(r.read_text())
    e = res["energy"]
    rows.append((d, res["dataset"], res["augment"], len(res["history"]["epoch_val_acc"]),
                 res["best_val_accuracy"], res["test"]["accuracy"],
                 res["wall_time_s"]["train"], (e.get("train") or {}).get("energy_j")))
print(f"{'run':38s} {'dataset':9s} {'aug':5s} {'ep':>3s} {'best_val':>9s} {'test':>7s} {'train_s':>8s} {'train_J':>9s}")
for d, ds, aug, ep, bv, ta, ts, ej in rows:
    print(f"{d:38s} {ds:9s} {str(aug):5s} {ep:3d} {bv:9.4f} {ta:7.4f} {ts:8.0f} {ej if ej is None else round(ej):>9}")
for d, *_ in rows:
    for p in ("loss.png", "accuracy.png"):
        f = pathlib.Path(d, "plots", p)
        if f.exists():
            print(f); display(Image(str(f)))

## 7. Download the results (Colab)

In [ ]:
import shutil
archive = shutil.make_archive("snn_cifar_runs", "zip", ".", "runs")
print("written:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass